# GPT-2 Generation

Compare the next-token distribution and sampled continuations. Is fluent text reliable?

Run the setup cells, then change the input in **Demo code**. All packages, models and data stay in the Colab runtime. For large models, select a GPU runtime. Free GPU availability varies.

Source: https://huggingface.co/openai-community/gpt2

## Runtime setup

In [ ]:
# Packages are installed in the Colab runtime, not on your computer.
%pip -q install "matplotlib>=3.9,<3.12" "scikit-learn==1.6.1" "Pillow" "requests" "tqdm" "transformers==4.57.1" "huggingface-hub==0.36.2" "accelerate" "safetensors" "qwen-vl-utils"
import os
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_MODE"] = "disabled"
import torch
print("Runtime:", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import os
import random
import tarfile
import requests
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,   # LLM
    Qwen3VLForConditionalGeneration,        # Multimodal LLM
    AutoProcessor,
)
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name()}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Demonstrate the next-token prediction principle with a small GPT-2 model
from transformers import GPT2LMHeadModel, GPT2Tokenizer

LLM_ID = 'gpt2'  # 124M — light and fast, for principle demonstration
print(f"📦 Loading LLM: {LLM_ID}")
llm_tokenizer = GPT2Tokenizer.from_pretrained(LLM_ID)
llm_model = GPT2LMHeadModel.from_pretrained(LLM_ID).to(device)
llm_model.eval()

params = sum(p.numel() for p in llm_model.parameters())
print(f"   ✅ Loaded! ({params/1e6:.0f}M params)")
print(f"   - Vocab size: {llm_tokenizer.vocab_size}")
print(f"   - Max length: {llm_model.config.n_positions}")

In [ ]:
@torch.no_grad()
def visualize_next_token(text, top_k=10):
    """Visualize the probability distribution of the next token."""
    inputs = llm_tokenizer(text, return_tensors='pt').to(device)
    outputs = llm_model(**inputs)
    
    # Logits at the last position → next-token probabilities
    next_logits = outputs.logits[0, -1, :]
    probs = torch.softmax(next_logits, dim=0)
    top_probs, top_ids = probs.topk(top_k)
    
    tokens = [llm_tokenizer.decode(idx.item()) for idx in top_ids]
    probs_list = top_probs.cpu().numpy()
    
    return tokens, probs_list

# Predict next token for several prompts
prompts = [
    "The dog is",
    "A cute little cat",
    "Dogs and cats are popular",
    "The golden retriever",
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('🔮 LLM Next Token Prediction — "What word comes next?"', fontsize=16, fontweight='bold')

colors = ['#4ECDC4', '#FF6B6B', '#45B7D1', '#96CEB4']

for i, prompt in enumerate(prompts):
    ax = axes[i // 2, i % 2]
    tokens, probs = visualize_next_token(prompt)
    
    # Clean up tokens for display (whitespace/special chars)
    display_tokens = [repr(t).strip("'") for t in tokens]
    
    bars = ax.barh(range(len(tokens)), probs, color=colors[i], alpha=0.8, edgecolor='black')
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(display_tokens, fontsize=10, fontfamily='monospace')
    ax.set_xlabel('Probability')
    ax.set_title(f'"{prompt}" + ???', fontsize=12, fontweight='bold')
    ax.invert_yaxis()
    
    for bar, p in zip(bars, probs):
        ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
               f'{p:.1%}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
def generate_text(prompt, max_new_tokens=60, temperature=1.0, top_k=50, top_p=0.9, do_sample=True):
    """Generate text with GPT-2."""
    inputs = llm_tokenizer(prompt, return_tensors='pt').to(device)
    
    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=do_sample,
            pad_token_id=llm_tokenizer.eos_token_id,
        )
    
    generated = llm_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated

# Compare temperatures
prompt = "Dogs are wonderful pets because"

print("🌡️ Temperature comparison\n")
print(f"  Prompt: \"{prompt}\"\n")

temperatures = [0.3, 0.7, 1.0, 1.5]
for temp in temperatures:
    result = generate_text(prompt, temperature=temp, max_new_tokens=50)
    new_text = result[len(prompt):].strip()
    print(f"  🌡️ T={temp}: {new_text[:120]}...")
    print()

print("💡 Low temperature → conservative and repetitive")
print("   High temperature → creative but can be off-topic")

## Demo code

Compare the next-token distribution and sampled continuations. Is fluent text reliable?

In [ ]:
prompt = "The dog looked at the"
completion = generate_text(prompt, max_new_tokens=40, temperature=0.7)
print(completion)

In [ ]:
# Visualize how temperature changes the probability distribution
prompt = "The cat sat on the"
inputs = llm_tokenizer(prompt, return_tensors='pt').to(device)
with torch.no_grad():
    logits = llm_model(**inputs).logits[0, -1, :]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle(f'🌡️ Effect of temperature on the probability distribution — "{prompt}" + ???',
            fontsize=14, fontweight='bold')

for i, temp in enumerate([0.1, 0.5, 1.0, 2.0]):
    probs = torch.softmax(logits / temp, dim=0)
    top_probs, top_ids = probs.topk(8)
    tokens = [repr(llm_tokenizer.decode(idx.item())).strip("'") for idx in top_ids]
    
    axes[i].bar(range(len(tokens)), top_probs.cpu().numpy(), color=plt.cm.coolwarm(temp/2.5),
               edgecolor='black', alpha=0.8)
    axes[i].set_xticks(range(len(tokens)))
    axes[i].set_xticklabels(tokens, rotation=45, ha='right', fontsize=9, fontfamily='monospace')
    axes[i].set_ylabel('Probability')
    axes[i].set_ylim(0, 1)
    axes[i].set_title(f'T = {temp}', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("💡 T=0.1: nearly greedy (mass concentrates on the top one) / T=2.0: nearly uniform (spread evenly)")

In [ ]:
# Generate Dog/Cat-related text (various prompts)
creative_prompts = [
    "A day in the life of a golden retriever:",
    "Why cats are the best pets:",
    "The difference between dogs and cats is that",
    "If my dog could talk, it would say:",
]

print("📝 LLM text generation (Dog/Cat topics)\n")
for p in creative_prompts:
    result = generate_text(p, temperature=0.8, max_new_tokens=60)
    print(f"  💬 Prompt: \"{p}\"")
    print(f"  📖 Generated: {result[:200]}")
    print()

## Interpretation

GPT-2 is used to demonstrate decoding, not as an instruction assistant.

Compare the output with the evidence in the input. A generated answer can be wrong.